# Inspect the completed CCR study

Extract CCR_Raw_Diagnostics.zip into /user/cbaker4/connectome, then open this notebook there. It adds this notebook and one new script. Keep the original study and environment.

Use Run All. This notebook checks saved trials in the foreground and makes the download only after success. It does not launch simulations. A starting allocation is 4 cores, 16000 MB, 2 hours, no GPU; runtime has not been measured on the full CCR dataset. Progress prints every 30 trials. Keep this session running until it prints the download name.

The existing .ccr-venv and ccr_python.json from the repaired Python setup are used. No package installation is needed. Analysis CPU details are recorded separately from the unchanged simulation provenance.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib, zipfile
ROOT = Path.cwd().resolve()
STUDY = ROOT / 'results/expanded_sensitivity'
OUT = ROOT / 'results/raw_diagnostics_20260927'
PYTHON = ROOT / '.ccr-venv/bin/python'
RUN_ENV = os.environ.copy()
config_path = ROOT / 'ccr_python.json'
if not config_path.is_file():
    raise FileNotFoundError('Missing saved CCR module configuration: ' + str(config_path))
config = json.loads(config_path.read_text())
for key, value in config.get('env', {}).items():
    if value is None:
        RUN_ENV.pop(key, None)
    else:
        RUN_ENV[key] = value
for key in ['PYTHONHOME', 'PYTHONPATH']:
    RUN_ENV.pop(key, None)
RUN_ENV.update(PYTHONNOUSERSITE='1', MPLBACKEND='Agg', OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
if not (ROOT / 'transfer_manifest.json').is_file() or not PYTHON.is_file():
    raise RuntimeError('Open this notebook in the original connectome directory with its existing environment.')
SCRIPT = ROOT / 'scripts/pcdr_raw_diagnostics.py'
if hashlib.sha256(SCRIPT.read_bytes()).hexdigest() != 'fc368a739831c708e0492380d1547abd4e3bcb700cd13e9a5bc10cfb8b45fdcc':
    raise RuntimeError('Diagnostic script differs from the reviewed upload.')
subprocess.run([str(PYTHON), '-c', 'import sys, numpy, pandas, pyarrow; assert sys.version_info[:2] == (3,11); print(sys.version)'], env=RUN_ENV, check=True)
print('Reading:', STUDY)
print('Writing:', OUT)


The script refuses an existing output directory. If a previous attempt failed, retain that folder as evidence and change OUT above to a new name. Do not delete study locks unless the original worker/controller has been checked and is no longer running. If the notebook disconnects, inspect OUT/audit.json before starting another attempt.


In [ ]:
command = [str(PYTHON), '-u', str(SCRIPT), '--study', str(STUDY), '--out', str(OUT)]
with subprocess.Popen(command, cwd=ROOT, env=RUN_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    result = process.wait()
if result:
    raise RuntimeError('Diagnostic failed; retain its audit.json and the error printed above.')
audit = json.loads((OUT / 'audit.json').read_text())
if audit['status'] != 'complete':
    raise RuntimeError('Diagnostics have not completed.')
print('Validated trials:', audit['completed_trials'])


In [ ]:
audit = json.loads((OUT / 'audit.json').read_text())
if audit['status'] != 'complete':
    raise RuntimeError('Do not package partial diagnostics.')
def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()
for name, expected in audit['outputs'].items():
    if sha256(OUT / name) != expected:
        raise RuntimeError('Changed diagnostic output: ' + name)
archive_path = ROOT / 'CCR_raw_diagnostics_results.zip'
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in ['audit.json'] + sorted(audit['outputs']):
        archive.write(OUT / name, 'raw_diagnostics/' + name)
print('Download:', archive_path.name)
print('Size (MB):', round(archive_path.stat().st_size / 1e6, 2))


The export includes all trial summaries and 10 ms population counts, sparse baseline rates and sparse paired rate changes. neurons.csv defines the full neuron universe: omitted sparse rows mean exact zero. trial_diagnostics.csv retains every trial, including all-zero changes. The fixed-mode absolute readout is a per-seed diagnostic, not the original mean-vector A/F endpoint. No large-response seeds are removed. No bootstrap or new biological conclusion is produced by this notebook.
